In [ ]:
import _pickle as pickle
import matplotlib.pyplot as plt
import os
import torch
import torch.optim as optim
import torch.nn.functional as F

from envs import TFAugmentedFrozenLakeEnv
from policies import ThoughtMLP

## Generate expert data

In [ ]:
num_cells = 16
save_path = "expert_data-tabular.pt"

config_path = "/home/bryanpu1/projects/thinking-as-control/EXPS/frozen_lake-ppo/no_thought_acts-tabular_42-args.pkl"
model_path = "/home/bryanpu1/projects/thinking-as-control/EXPS/frozen_lake-ppo/no_thought_acts-tabular_42.pt"

In [ ]:
if os.path.isfile(save_path):
    data_dict = pickle.load(open(save_path, "rb"))
    states = data_dict["obss"]
    actions = data_dict["acts"]
else:
    config = pickle.load(open(config_path, "rb"))
    policy = torch.load(model_path, weights_only=False)
    d_model = config.d_model

    states = torch.eye(num_cells)

    logits, _ = policy(torch.cat((
        states,
        torch.zeros((num_cells, d_model)),
    ), dim=1))
    actions = torch.argmax(logits, dim=-1)

    pickle.dump({
        "obss": states,
        "acts": actions,
    }, open(save_path, "wb"))

## Behavioural cloning

In [ ]:
n_thought_states = 1
n_thought_acts = 0
max_steps = 1
tabular = False

lr = 1e-2
num_epochs = 100

In [ ]:
env = TFAugmentedFrozenLakeEnv(
        n_thought_states=n_thought_states,
        n_thought_acts=n_thought_acts,
        d_model=d_model,
        max_steps=max_steps,
        tabular=tabular,
    )

policy_optimizer = optim.Adam(policy.parameters(), lr=lr, weight_decay=0.0)
policy = ThoughtMLP(
    obs_dim=env.obs_dim,
    n_acts=env.n_acts,
    n_thought_acts=n_thought_acts,
    d_model=d_model,
)

enc_obss = torch.cat((
    torch.argmax(states, dim=-1, keepdims=True),
    torch.zeros((num_cells, d_model)),
), dim=1)

In [ ]:
losses = []
for _ in range(num_epochs):
    logits, _ = policy(enc_obss)
    loss = F.cross_entropy(logits, actions).mean()
    policy_optimizer.zero_grad()
    loss.backward()
    policy_optimizer.step()
    losses.append(loss.item())

In [ ]:
plt.plot(range(num_epochs), losses)

In [ ]:
logits, _ = policy(enc_obss)
pred_acts = torch.argmax(logits, dim=-1)
print((pred_acts == actions).mean())